# Week 3: Advanced AI for Clinical Reality
**FURP 2026 — Cancer Detection Using Artificial Intelligence**  
**Instructor: Prof. Elio Espejo**

---

## 本周目标
1. 不确定性量化（Monte Carlo Dropout）
2. 类别不平衡处理（5% 癌症真实场景）
3. AI Assistant Tasks：迁移学习 / 高级数据增强 / 集成方法 / Grad-CAM 可解释性

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             confusion_matrix, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{device}')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# 复用数据生成和模型（从Week2）
def create_synthetic_mammograms(num_samples=500):
    np.random.seed(42)
    images, labels = [], []
    image_size = 256
    y, x = np.ogrid[:image_size, :image_size]
    for i in range(num_samples):
        image = np.random.normal(120, 25, (image_size, image_size))
        image += 20 * np.sin(x/30) * np.cos(y/25)
        has_cancer = np.random.random() < 0.3
        if has_cancer:
            cx = np.random.randint(40, image_size-40)
            cy = np.random.randint(40, image_size-40)
            r  = np.random.randint(15, 25)
            for dy in range(-r, r):
                for dx in range(-r, r):
                    if dx**2+dy**2 < r**2 and 0<=cy+dy<image_size and 0<=cx+dx<image_size:
                        image[cy+dy, cx+dx] += 60 + np.random.normal(0, 5)
            labels.append(1)
        else:
            labels.append(0)
        image = np.clip(image + np.random.normal(0,10,image.shape), 0, 255).astype(np.uint8)
        images.append(image)
    return np.array(images), np.array(labels)

class MedicalCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256,128), nn.ReLU(True), nn.Dropout(0.3), nn.Linear(128,num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0),-1))

class MedicalImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images=images.astype(np.uint8); self.labels=labels.astype(np.int64); self.transform=transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx], mode='L')
        if self.transform: img = self.transform(img)
        else: img = torch.FloatTensor(np.array(img)).unsqueeze(0)/255.0
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

val_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([0.5],[0.5])
])

images, labels = create_synthetic_mammograms(500)
print(f'数据集：{len(images)}张 | 癌症:{np.sum(labels)} | 正常:{np.sum(labels==0)}')

---
## Step 1: 不确定性量化（Monte Carlo Dropout）

**原理：**  
Dropout 训练时随机关闭神经元，推断时通常关闭 Dropout。  
MC Dropout 推断时保持 Dropout 激活，多次前向传播 → 预测分布：

$$P(y|x) \approx \frac{1}{T}\sum_{t=1}^{T} p_\theta^{\hat{t}}(y|x)$$

标准差衡量不确定性：**不确定性高 → 需要专家复审**

In [ ]:
def predict_with_uncertainty(model, image_tensor, num_samples=50):
    """Monte Carlo Dropout 不确定性估计"""
    def enable_dropout(m):
        if isinstance(m, nn.Dropout):
            m.train()  # 推断时保持Dropout激活

    model.eval()
    model.apply(enable_dropout)

    predictions = []
    with torch.no_grad():
        for _ in range(num_samples):
            output = model(image_tensor)
            probs  = F.softmax(output, dim=1)
            predictions.append(probs.cpu().numpy())

    predictions = np.array(predictions)  # (T, batch, 2)
    mean_pred   = np.mean(predictions, axis=0)
    uncertainty = np.std(predictions, axis=0)

    cancer_prob = mean_pred[0, 1]
    cancer_unc  = uncertainty[0, 1]

    if cancer_unc > 0.3:
        recommendation = 'HIGH_UNCERTAINTY — 需要专家复审'
        confidence = 'LOW'
    elif cancer_prob > 0.7:
        recommendation = 'SUSPICIOUS — 建议紧急随访'
        confidence = 'HIGH' if cancer_unc < 0.15 else 'MEDIUM'
    elif cancer_prob > 0.3:
        recommendation = 'BORDERLINE — 建议追加影像'
        confidence = 'MEDIUM'
    else:
        recommendation = 'NORMAL — 继续常规筛查'
        confidence = 'HIGH' if cancer_unc < 0.15 else 'MEDIUM'

    return {
        'cancer_probability': float(cancer_prob),
        'uncertainty':        float(cancer_unc),
        'recommendation':     recommendation,
        'confidence':         confidence,
        'requires_expert':    cancer_unc > 0.3,
        'all_predictions':    predictions[:, 0, 1]
    }


# 测试：对不同图像做不确定性分析
model = MedicalCNN().to(device)

sample_images = {
    '正常图像':    images[np.where(labels==0)[0][0]],
    '癌症图像':    images[np.where(labels==1)[0][0]],
    '嘈杂图像':    np.clip(images[0].astype(float) + np.random.normal(0,50,images[0].shape), 0, 255).astype(np.uint8)
}

print('不确定性分析结果：')
print('='*60)
results_unc = {}
for name, img in sample_images.items():
    tensor = val_transform(Image.fromarray(img, mode='L')).unsqueeze(0).to(device)
    result = predict_with_uncertainty(model, tensor, num_samples=50)
    results_unc[name] = result
    print(f'\n{name}：')
    print(f'  癌症概率：{result["cancer_probability"]:.3f}')
    print(f'  不确定性：{result["uncertainty"]:.3f}')
    print(f'  置信度：  {result["confidence"]}')
    print(f'  建议：    {result["recommendation"]}')

In [ ]:
# 不确定性分布可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Monte Carlo Dropout: Prediction Distributions', fontsize=15, fontweight='bold')

colors_map = {'正常图像': '#1D9E75', '癌症图像': '#E24B4A', '嘈杂图像': '#EF9F27'}
for ax, (name, result) in zip(axes, results_unc.items()):
    preds = result['all_predictions']
    ax.hist(preds, bins=20, color=colors_map[name], alpha=0.8, edgecolor='white')
    ax.axvline(result['cancer_probability'], color='black', lw=2,
               label=f'Mean={result["cancer_probability"]:.3f}')
    ax.axvline(result['cancer_probability']-result['uncertainty'],
               color='gray', lw=1.5, linestyle='--')
    ax.axvline(result['cancer_probability']+result['uncertainty'],
               color='gray', lw=1.5, linestyle='--', label=f'σ={result["uncertainty"]:.3f}')
    ax.set_xlabel('Cancer Probability', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{name}\n{result["confidence"]} confidence', fontweight='bold', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week3_uncertainty.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 2: 类别不平衡处理（真实临床场景）

**真实场景：** 乳腺癌筛查中约 3-5% 阳性率  

**处理方法：**
1. 加权损失函数：$w_c = \frac{N}{2 \times N_c}$
2. 过采样（SMOTE 等）
3. 调整决策阈值

In [ ]:
def create_imbalanced_dataset(images, labels, cancer_rate=0.05):
    """创建真实临床比例的不平衡数据集"""
    normal_idx = np.where(labels == 0)[0]
    cancer_idx = np.where(labels == 1)[0]
    n_cancer = min(25, len(cancer_idx))
    n_normal = int(n_cancer / cancer_rate)
    n_normal = min(n_normal, len(normal_idx))
    sel_normal = np.random.choice(normal_idx, n_normal, replace=False)
    sel_cancer = np.random.choice(cancer_idx, n_cancer, replace=False)
    idx = np.concatenate([sel_normal, sel_cancer])
    np.random.shuffle(idx)
    imgs_imb = images[idx]
    labs_imb = labels[idx]
    actual_rate = np.mean(labs_imb)
    print(f'不平衡数据集：{len(imgs_imb)}张 | 癌症:{np.sum(labs_imb)} ({actual_rate:.1%}) | 正常:{np.sum(labs_imb==0)}')
    return imgs_imb, labs_imb

imgs_imb, labs_imb = create_imbalanced_dataset(images, labels, cancer_rate=0.05)

# 计算类别权重
cw = compute_class_weight('balanced', classes=np.unique(labs_imb), y=labs_imb)
cw_tensor = torch.FloatTensor(cw).to(device)
print(f'类别权重：正常={cw[0]:.2f}  癌症={cw[1]:.2f}')
print(f'癌症样本权重是正常的 {cw[1]/cw[0]:.1f} 倍')

# 可视化类别不平衡
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Class Imbalance: Balanced vs Real Clinical', fontsize=14, fontweight='bold')

for ax, (title, lbs) in zip(axes, [
    ('Week 1/2 数据集（均衡）', labels),
    ('真实临床场景（5% 癌症）', labs_imb)
]):
    counts = [np.sum(lbs==0), np.sum(lbs==1)]
    ax.pie(counts, labels=['Normal', 'Cancer'],
           colors=['#1D9E75','#E24B4A'],
           autopct='%1.1f%%', startangle=90,
           explode=(0, 0.1))
    ax.set_title(title, fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('week3_imbalance.png', bbox_inches='tight', dpi=150)
plt.show()

---
## AI Assistant Task 1: 迁移学习 + 逐步解冻

**逐步解冻策略：**
1. 第一阶段：冻结全部预训练层，只训练分类头
2. 第二阶段：解冻最后几层，用更小学习率微调
3. 第三阶段：全部解冻，整体微调

In [ ]:
import torchvision.models as models

class ProgressiveFineTuneCNN(nn.Module):
    """
    逐步解冻迁移学习
    数学原理：不同层的学习率不同（差分学习率）
    低层：lr × 0.01（保持通用特征）
    高层：lr × 1.0 （快速适应医学特征）
    """
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_feat = resnet.fc.in_features
        resnet.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_feat, 128),
            nn.ReLU(True), nn.Linear(128, 2)
        )
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.head = resnet.fc

    def forward(self, x):
        x = self.backbone(x).view(x.size(0), -1)
        return self.head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print('阶段1：冻结backbone，只训练分类头')

    def unfreeze_top(self, n_layers=2):
        layers = list(self.backbone.children())
        for layer in layers[-n_layers:]:
            for p in layer.parameters(): p.requires_grad = True
        print(f'阶段2：解冻最后{n_layers}层')

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True
        print('阶段3：全部解冻')


tl_model = ProgressiveFineTuneCNN().to(device)

# 展示三个阶段的参数统计
stages = []
for stage_name, method in [
    ('阶段1：冻结backbone', tl_model.freeze_backbone),
    ('阶段2：解冻Top2层', lambda: tl_model.unfreeze_top(2)),
    ('阶段3：全部解冻', tl_model.unfreeze_all)
]:
    method()
    trainable = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in tl_model.parameters())
    stages.append((stage_name, trainable, total))
    print(f'  可训练参数：{trainable:,} / {total:,} ({trainable/total*100:.1f}%)')

# 可视化
fig, ax = plt.subplots(figsize=(10, 4))
stage_names = [s[0] for s in stages]
trainable_p = [s[1]/1000 for s in stages]
frozen_p    = [(s[2]-s[1])/1000 for s in stages]
x = np.arange(len(stages))
ax.bar(x, trainable_p, label='可训练参数 (K)', color='#1D9E75', alpha=0.85)
ax.bar(x, frozen_p, bottom=trainable_p, label='冻结参数 (K)', color='#B4B2A9', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(stage_names, fontsize=10)
ax.set_ylabel('参数量 (K)', fontsize=12)
ax.set_title('Progressive Fine-tuning: Trainable Parameters per Stage', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('task1_progressive_finetuning.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 1 完成！')

---
## AI Assistant Task 2: 医学专用数据增强

**关键原则：** 增强必须保留诊断内容！
- 弹性变形：模拟患者体位差异
- 强度归一化：统一不同设备的成像风格
- 噪声模拟：模拟X射线量子噪声

In [ ]:
from scipy.ndimage import gaussian_filter, map_coordinates

def elastic_deformation(image, alpha=50, sigma=5):
    """
    弹性变形：模拟不同体位
    数学原理：用高斯平滑的随机位移场变形图像
    dx(x,y) = alpha × GaussianFilter(rand(-1,1))
    """
    shape = image.shape
    dx = gaussian_filter((np.random.rand(*shape)*2-1), sigma) * alpha
    dy = gaussian_filter((np.random.rand(*shape)*2-1), sigma) * alpha
    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    indices = np.reshape(y+dy, (-1,1)), np.reshape(x+dx, (-1,1))
    return map_coordinates(image, indices, order=1, mode='reflect').reshape(shape).astype(np.uint8)

def clahe_normalization(image):
    """CLAHE 对比度增强"""
    import cv2
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(image)

def add_quantum_noise(image, noise_level=15):
    """泊松分布量子噪声（X射线物理特性）"""
    noisy = image.astype(float) + np.random.normal(0, noise_level, image.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

# 可视化增强效果
sample = images[np.where(labels==1)[0][0]]
augmented = {
    '原图':     sample,
    '弹性变形': elastic_deformation(sample, alpha=30, sigma=4),
    'CLAHE':    clahe_normalization(sample),
    '量子噪声': add_quantum_noise(sample, noise_level=20),
    '旋转15°':  np.array(Image.fromarray(sample).rotate(15)),
    '组合增强': add_quantum_noise(clahe_normalization(
                    elastic_deformation(sample, alpha=20, sigma=4)), 10)
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Medical-Specific Data Augmentation', fontsize=15, fontweight='bold')
for ax, (name, img) in zip(axes.flatten(), augmented.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('task2_medical_augmentation.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 2 完成！')

---
## AI Assistant Task 3: 集成方法（Ensemble）

**原理：** 多个模型的预测融合，误差相互抵消：
$$P_{\text{ensemble}}(y|x) = \frac{1}{M}\sum_{m=1}^{M} P_m(y|x)$$

集成可以降低方差，提高稳定性。

In [ ]:
# 训练多个模型（不同随机种子）
def quick_train(images, labels, seed=42, epochs=5):
    torch.manual_seed(seed)
    X_tr, X_te, y_tr, y_te = train_test_split(
        images, labels, test_size=0.2, random_state=seed, stratify=labels)
    transform = transforms.Compose([
        transforms.Resize((64,64)), transforms.ToTensor(), transforms.Normalize([0.5],[0.5])
    ])
    loader = DataLoader(MedicalImageDataset(X_tr, y_tr, transform), batch_size=16, shuffle=True)
    test_loader = DataLoader(MedicalImageDataset(X_te, y_te, transform), batch_size=16)

    # 小模型快速训练
    m = nn.Sequential(
        nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
        nn.Flatten(), nn.Linear(32,2)
    ).to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    w = torch.FloatTensor([1.0, len(y_tr)/np.sum(y_tr==1)]).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    for ep in range(epochs):
        m.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(m(x), y).backward(); opt.step()
    m.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for x, y in test_loader:
            all_p.extend(F.softmax(m(x.to(device)),dim=1)[:,1].cpu().numpy())
            all_y.extend(y.numpy())
    return np.array(all_p), np.array(all_y)

print('训练5个集成模型（不同随机种子）...')
ensemble_probs = []
ensemble_labels = None
for seed in [42, 123, 456, 789, 1024]:
    probs, labs = quick_train(images, labels, seed=seed, epochs=5)
    ensemble_probs.append(probs)
    ensemble_labels = labs
    auc_s = roc_auc_score(labs, probs)
    print(f'  模型 seed={seed}: AUC={auc_s:.3f}')

ensemble_probs = np.array(ensemble_probs)
# 简单平均融合
avg_probs = np.mean(ensemble_probs, axis=0)
print(f'\n集成后 AUC: {roc_auc_score(ensemble_labels, avg_probs):.3f}')

# 可视化各模型和集成结果
fig, ax = plt.subplots(figsize=(10, 5))
for i, probs in enumerate(ensemble_probs):
    fpr, tpr, _ = roc_curve(ensemble_labels, probs)
    ax.plot(fpr, tpr, alpha=0.5, lw=1.5,
            label=f'Model {i+1} AUC={roc_auc_score(ensemble_labels,probs):.3f}')
fpr_e, tpr_e, _ = roc_curve(ensemble_labels, avg_probs)
ax.plot(fpr_e, tpr_e, 'k-', lw=3,
        label=f'Ensemble AUC={roc_auc_score(ensemble_labels,avg_probs):.3f}')
ax.plot([0,1],[0,1],'--',color='gray',lw=1.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Ensemble vs Individual Models: ROC Curves', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('task3_ensemble.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 3 完成！')

---
## AI Assistant Task 4: Grad-CAM 可解释性

**原理：** 计算目标类别的梯度对最后卷积层的加权激活图：
$$\text{Grad-CAM} = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right), \quad \alpha_k^c = \frac{1}{Z}\sum_{i,j}\frac{\partial y^c}{\partial A_{ij}^k}$$

亮的区域 = CNN 做决策时关注的区域

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=1):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward()
        # 全局平均池化梯度
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        # 加权激活图
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        # 归一化到 [0, 1]
        cam -= cam.min(); cam /= (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()


# 对癌症图像生成 Grad-CAM
model_gc = MedicalCNN().to(device)
model_gc.eval()
gradcam = GradCAM(model_gc, model_gc.features[8])  # Block3 卷积层

cancer_img = images[np.where(labels==1)[0][0]]
normal_img = images[np.where(labels==0)[0][0]]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Grad-CAM: Where CNN Looks for Cancer', fontsize=15, fontweight='bold')

for row, (img, title) in enumerate([(cancer_img, '癌症图像'), (normal_img, '正常图像')]):
    tensor = val_transform(Image.fromarray(img, mode='L')).unsqueeze(0).to(device)
    cam = gradcam.generate(tensor, class_idx=1)
    cam_resized = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
        (img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0

    axes[row][0].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[row][0].set_title(f'{title}\n原图', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(cam_resized, cmap='jet')
    axes[row][1].set_title('Grad-CAM 热力图\n（红色=CNN关注区域）', fontsize=10)
    axes[row][1].axis('off')

    overlay = np.stack([img/255]*3, axis=-1)
    heatmap = plt.cm.jet(cam_resized)[:,:,:3]
    overlaid = 0.6*overlay + 0.4*heatmap
    axes[row][2].imshow(np.clip(overlaid,0,1))
    axes[row][2].set_title('叠加图', fontsize=10)
    axes[row][2].axis('off')

plt.tight_layout()
plt.savefig('task4_gradcam.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 4 完成！Grad-CAM 揭示了CNN的决策依据')

---
## Week 3 总结

| 技术 | 解决的问题 | 临床意义 |
|------|-----------|----------|
| MC Dropout | 不确定性估计 | 自动标记需人工复审的疑难病例 |
| 加权损失 | 类别不平衡 | 适应真实5%阳性率场景 |
| 逐步解冻 | 迁移学习 | 用少量医学数据获得好效果 |
| 弹性变形 | 数据增强 | 模拟真实体位差异 |
| 集成方法 | 减少方差 | 提升预测稳定性 |
| Grad-CAM | 可解释性 | 帮助医生理解AI决策依据 |

**下周预告：** 临床部署 — 从研究原型到医院系统！